# Random Spectrograms Viewer — ASVspoof5
Pick a split (`train` or `dev`) and optionally filter by label (`spoof` / `bonafide`). Five random utterances are sampled from the manifest, extracted from the tar archives, and displayed as mel-spectrograms.

In [ ]:
DATA_ROOT = "/Users/edenadiv/Eden's Files/Shenkar BSc/Final Project git/BioVoice/data/ASV5spoof5_tars"

SPLIT = "train"          # "train" or "dev"
LABEL_FILTER = None       # None = any, or "spoof" / "bonafide"
NUM_SAMPLES = 5

In [ ]:
import io
import tarfile
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── 1. Load manifest ──────────────────────────────────────────────
root = Path(DATA_ROOT)
protocols_dir = root / "ASVspoof5_protocols"

MANIFEST_COLS = [
    "speaker_id", "utt_id", "gender", "x1", "x2", "x3",
    "codec", "system_id", "label", "x4",
]

if SPLIT == "train":
    manifest_file = protocols_dir / "ASVspoof5.train.tsv"
    tar_dir = root / "ASVspoof5_audio_train_tars"
    tar_glob = "flac_T_*.tar"
elif SPLIT == "dev":
    manifest_file = protocols_dir / "ASVspoof5.dev.track_1.tsv"
    tar_dir = root / "ASVspoof5_audio_dev_tars"
    tar_glob = "flac_D_*.tar"
else:
    raise ValueError(f"SPLIT must be 'train' or 'dev', got '{SPLIT}'")

df = pd.read_csv(manifest_file, sep=r"\s+", header=None, names=MANIFEST_COLS)
print(f"Manifest: {len(df)} utterances  |  labels: {df['label'].value_counts().to_dict()}")

if LABEL_FILTER:
    df = df[df["label"] == LABEL_FILTER].reset_index(drop=True)
    print(f"After filter ('{LABEL_FILTER}'): {len(df)} utterances")

# ── 2. Build full tar index (handles truncated tars) ──────────────
#   Stores TarInfo objects so we can extract without calling getmember()
print("Building tar index (this may take a moment)...")
tar_index: dict[str, tuple[Path, tarfile.TarInfo]] = {}

for tar_path in sorted(tar_dir.glob(tar_glob)):
    with tarfile.open(tar_path, "r", ignore_zeros=True) as tf:
        while True:
            try:
                member = tf.next()
                if member is None:
                    break
                if member.isfile():
                    utt_id = Path(member.name).stem
                    tar_index[utt_id] = (tar_path, member)
            except tarfile.ReadError:
                break

print(f"Indexed {len(tar_index)} audio files from tar archives")

# ── 3. Filter manifest to available utterances and sample ─────────
df = df[df["utt_id"].isin(tar_index)].reset_index(drop=True)
print(f"Available in tars: {len(df)} utterances")

samples = df.sample(n=min(NUM_SAMPLES, len(df)), random_state=None)
print(f"\nSelected utterances:")
print(samples[["speaker_id", "utt_id", "label", "system_id"]].to_string(index=False))

# ── 4. Extract audio and plot spectrograms ─────────────────────────
fig, axes = plt.subplots(len(samples), 1, figsize=(14, 4 * len(samples)))
if len(samples) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, samples.iterrows()):
    utt_id = row["utt_id"]
    tar_path, member_info = tar_index[utt_id]
    with tarfile.open(tar_path, "r", ignore_zeros=True) as tf:
        raw = tf.extractfile(member_info).read()

    waveform, sr = librosa.load(io.BytesIO(raw), sr=16000, mono=True)
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=sr, n_mels=64, n_fft=512, hop_length=256, power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    librosa.display.specshow(mel_db, sr=sr, hop_length=256, x_axis="time", y_axis="mel", ax=ax)
    title = f"{utt_id}  |  speaker={row['speaker_id']}  |  label={row['label']}"
    if row["label"] == "spoof":
        title += f"  |  system={row['system_id']}"
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")

plt.tight_layout()
plt.show()